# 06 — Hyperparameter Tuning

**Joint notebook**, same CV harness and folds as notebook 05. Tune with `RandomizedSearchCV` (`scoring="average_precision"`, `random_state=RANDOM_STATE`), about 30-50 iterations.

| Owner | Model | Tuning focus |
| --- | --- | --- |
| Meegasthanna | Logistic Regression | `C`, `penalty`, `class_weight` |
| Bandara | XGBoost / LightGBM | `n_estimators`, `max_depth`, `learning_rate`, `scale_pos_weight` |
| Seneviratne | Random Forest | `n_estimators`, `max_depth`, `min_samples_leaf`, `class_weight` |
| Umer | SVM (RBF) | `C`, `gamma`, `class_weight` |

Umer also owns the SMOTE-vs-class-weights ablation and decision-threshold tuning (cross-cutting, applies to whichever model ends up selected).

**Output:** the results table from 05 extended with tuned rows (baseline vs tuned comparison).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [ ]:
import json

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import xgboost as xgb

from src.config import RANDOM_STATE, PROCESSED_DATA_DIR
from src.pipeline import build_preprocessing_pipeline
from src.evaluate import evaluate_cv, METRIC_NAMES

## Load the same train split and CV folds as notebook 05

TODO: reuse the exact same `StratifiedGroupKFold` configuration (same `random_state`, same `n_splits`) so baseline and tuned results are comparable.

In [3]:
train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train.parquet")

with open(PROCESSED_DATA_DIR / "eligible_features.json") as f:
    eligible_features = json.load(f)

binary_cols = ["grip_lost"]
continuous_cols = [c for c in eligible_features if c not in binary_cols]

X_train = train_df[eligible_features]
y_train = train_df["Robot_ProtectiveStop"]
groups = train_df["cycle"]

# Same fold configuration as notebook 05, so baseline and tuned results are comparable
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_train.shape, y_train.mean()

((5468, 16), 0.03108997805413314)

## Results table (baseline vs tuned)

TODO: load the baseline rows from notebook 05's output (re-run, or save/reload results as a small artifact in `data/processed/`), then append a tuned row per model here.

In [ ]:
result_columns = ["model", "stage"] + [f"{m}_mean" for m in METRIC_NAMES] + [f"{m}_std" for m in METRIC_NAMES] + ["best_params"]
results = pd.DataFrame(columns=result_columns)

def add_result(model_name, stage, scores, best_params=None):
    row = {"model": model_name, "stage": stage, "best_params": best_params, **scores}
    results.loc[len(results)] = row

# Seed with the baseline rows from notebook 05 (pinned values, same evaluate_cv() output format --
# notebook 06 doesn't re-run the baselines itself, so these should be refreshed if 05 is re-run with changes)
add_result("Dummy", "baseline", {
    "pr_auc_mean": 0.031932, "pr_auc_std": 0.005905,
    "recall_mean": 0.037326, "recall_std": 0.021369,
    "precision_mean": 0.036955, "precision_std": 0.023863,
    "f1_mean": 0.036773, "f1_std": 0.022340,
    "balanced_accuracy_mean": 0.504153, "balanced_accuracy_std": 0.010333,
    "roc_auc_mean": 0.504153, "roc_auc_std": 0.010333,
})
add_result("Logistic Regression", "baseline", {
    "pr_auc_mean": 0.111114, "pr_auc_std": 0.028067,
    "recall_mean": 0.671541, "recall_std": 0.094414,
    "precision_mean": 0.065405, "precision_std": 0.014887,
    "f1_mean": 0.118519, "f1_std": 0.024752,
    "balanced_accuracy_mean": 0.681313, "balanced_accuracy_std": 0.037009,
    "roc_auc_mean": 0.736851, "roc_auc_std": 0.068731,
})
add_result("XGBoost", "baseline", {
    "pr_auc_mean": 0.418357, "pr_auc_std": 0.037234,
    "recall_mean": 0.307251, "recall_std": 0.133273,
    "precision_mean": 0.551728, "precision_std": 0.131464,
    "f1_mean": 0.361558, "f1_std": 0.133408,
    "balanced_accuracy_mean": 0.648908, "balanced_accuracy_std": 0.064885,
    "roc_auc_mean": 0.911487, "roc_auc_std": 0.053311,
})
add_result("Random Forest", "baseline", {
    "pr_auc_mean": 0.428161, "pr_auc_std": 0.065739,
    "recall_mean": 0.161404, "recall_std": 0.116410,
    "precision_mean": 0.478959, "precision_std": 0.254741,
    "f1_mean": 0.230981, "f1_std": 0.148579,
    "balanced_accuracy_mean": 0.579038, "balanced_accuracy_std": 0.057255,
    "roc_auc_mean": 0.914904, "roc_auc_std": 0.055305,
})
add_result("SVM (RBF)", "baseline", {
    "pr_auc_mean": 0.254491, "pr_auc_std": 0.038116,
    "recall_mean": 0.732430, "recall_std": 0.149087,
    "precision_mean": 0.134044, "precision_std": 0.039397,
    "f1_mean": 0.223520, "f1_std": 0.058394,
    "balanced_accuracy_mean": 0.788816, "balanced_accuracy_std": 0.063932,
    "roc_auc_mean": 0.840530, "roc_auc_std": 0.074577,
})

results

## Logistic Regression tuning — Meegasthanna

In [ ]:
logreg_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver="liblinear")),
])

logreg_param_dist = {
    "classifier__C": loguniform(1e-4, 1e2),
    "classifier__penalty": ["l1", "l2"],
    "classifier__class_weight": ["balanced", None, {0: 1, 1: 10}, {0: 1, 1: 25}, {0: 1, 1: 32}]
}

logreg_search = RandomizedSearchCV(
    estimator=logreg_pipeline,
    param_distributions=logreg_param_dist,
    n_iter=35,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

logreg_search.fit(X_train, y_train, groups=groups)
print(f"Best CV PR-AUC Score: {logreg_search.best_score_:.4f}")
print("Best Hyperparameters:", logreg_search.best_params_)

# Evaluate tuned pipeline across the standard 5-fold CV harness
logreg_tuned_scores = evaluate_cv(logreg_search.best_estimator_, X_train, y_train, groups)
add_result("Logistic Regression", "tuned", logreg_tuned_scores, logreg_search.best_params_)

results

## XGBoost / LightGBM tuning — Bandara

In [ ]:
xgb_param_dist = {
    "classifier__n_estimators": randint(100, 500),
    "classifier__max_depth": randint(3, 10),
    "classifier__learning_rate": uniform(0.01, 0.29),
    "classifier__scale_pos_weight": uniform(1, 30),
}

xgb_search_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="aucpr")),
])

xgb_search = RandomizedSearchCV(
    xgb_search_pipeline, xgb_param_dist, n_iter=40,
    scoring="average_precision", cv=cv, random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_search.fit(X_train, y_train, groups=groups)

print("Best CV PR-AUC (search):", xgb_search.best_score_)
print("Best params:", xgb_search.best_params_)

# Re-run through evaluate_cv with the best params for the full metric set (mean+std),
# so this row is directly comparable to the baseline row's format.
best_xgb_params = {k.replace("classifier__", ""): v for k, v in xgb_search.best_params_.items()}
xgb_tuned_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="aucpr", **best_xgb_params)),
])
xgb_tuned_scores = evaluate_cv(xgb_tuned_pipeline, X_train, y_train, groups)
add_result("XGBoost", "tuned", xgb_tuned_scores, best_params=best_xgb_params)
results

## Random Forest tuning — Seneviratne

In [ ]:
rf_param_dist = {
    "classifier__n_estimators": randint(100, 500),
    "classifier__max_depth": randint(3, 20),
    "classifier__min_samples_leaf": randint(1, 20),
    "classifier__class_weight": ["balanced", "balanced_subsample"],
}

rf_search_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE)),
])

rf_search = RandomizedSearchCV(
    rf_search_pipeline, rf_param_dist, n_iter=40,
    scoring="average_precision", cv=cv, random_state=RANDOM_STATE, n_jobs=-1,
)
rf_search.fit(X_train, y_train, groups=groups)

print("Best CV PR-AUC (search):", rf_search.best_score_)
print("Best params:", rf_search.best_params_)

best_rf_params = {k.replace("classifier__", ""): v for k, v in rf_search.best_params_.items()}
rf_tuned_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, **best_rf_params)),
])
rf_tuned_scores = evaluate_cv(rf_tuned_pipeline, X_train, y_train, groups)
add_result("Random Forest", "tuned", rf_tuned_scores, best_params=best_rf_params)
results

## SVM (RBF) tuning — Umer

In [ ]:
svm_param_dist = {
    "classifier__C": loguniform(1e-2, 1e2),
    "classifier__gamma": loguniform(1e-4, 1e0),
    "classifier__class_weight": ["balanced", None],
}

svm_search_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
])

svm_search = RandomizedSearchCV(
    svm_search_pipeline, svm_param_dist, n_iter=40,
    scoring="average_precision", cv=cv, random_state=RANDOM_STATE, n_jobs=1,
)
svm_search.fit(X_train, y_train, groups=groups)
print("Best CV PR-AUC (search):", svm_search.best_score_)
print("Best params:", svm_search.best_params_)

# Re-run through evaluate_cv with the best params for the full metric set (mean+std)
best_svm_params = {k.replace("classifier__", ""): v for k, v in svm_search.best_params_.items()}
svm_tuned_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE, **best_svm_params)),
])
svm_tuned_scores = evaluate_cv(svm_tuned_pipeline, X_train, y_train, groups)
add_result("SVM (RBF)", "tuned", svm_tuned_scores, best_svm_params)
results

## SMOTE vs class weights — Umer

TODO: for the leading model(s), compare `class_weight="balanced"` against SMOTE-resampling the training folds only (never the validation fold — resample inside the CV loop, not before it, or it leaks).

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# --- XGBoost with SMOTE (leading model, tuned PR-AUC 0.473) ---
# Using imblearn.pipeline.Pipeline so SMOTE resamples only within each CV fold's
# training split -- evaluate_cv() clones and fits fresh per fold, so synthetic
# points from SMOTE never leak into the validation fold.
smote_xgb_pipeline = ImbPipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="aucpr")),
])
smote_xgb_scores = evaluate_cv(smote_xgb_pipeline, X_train, y_train, groups)
add_result("XGBoost + SMOTE", "ablation", smote_xgb_scores)

# --- XGBoost with SMOTE + tuned hyperparams ---
smote_xgb_tuned_pipeline = ImbPipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", xgb.XGBClassifier(
        random_state=RANDOM_STATE, eval_metric="aucpr",
        **{k: v for k, v in best_xgb_params.items() if k != 'scale_pos_weight'},
    )),
])
smote_xgb_tuned_scores = evaluate_cv(smote_xgb_tuned_pipeline, X_train, y_train, groups)
add_result("XGBoost + SMOTE (tuned)", "ablation", smote_xgb_tuned_scores)

# Compare class_weight="balanced" (already in results as tuned XGBoost) vs SMOTE
print('=== SMOTE vs class weights ablation ===')
print(f'XGBoost tuned (scale_pos_weight):  PR-AUC={xgb_tuned_scores["pr_auc_mean"]:.3f}, recall={xgb_tuned_scores["recall_mean"]:.3f}')
print(f'XGBoost + SMOTE (default):         PR-AUC={smote_xgb_scores["pr_auc_mean"]:.3f}, recall={smote_xgb_scores["recall_mean"]:.3f}')
print(f'XGBoost + SMOTE (tuned, no SPW):   PR-AUC={smote_xgb_tuned_scores["pr_auc_mean"]:.3f}, recall={smote_xgb_tuned_scores["recall_mean"]:.3f}')

results

## Decision-threshold tuning — Umer

TODO: sweep the classification threshold on validation-fold predictions (not test) and pick the operating point that best trades recall vs false-alarm rate for this safety use case.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import recall_score, precision_score, f1_score, balanced_accuracy_score

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Collect out-of-fold predictions from the best model (tuned XGBoost)
# to sweep thresholds on validation data (never test data)
xgb_best_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", xgb.XGBClassifier(
        random_state=RANDOM_STATE, eval_metric="aucpr", **best_xgb_params
    )),
])

oof_probas = np.zeros(len(y_train))
oof_preds_05 = np.zeros(len(y_train))
cv_threshold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for train_idx, val_idx in cv_threshold.split(X_train, y_train, groups):
    Xt, Xv = X_train.iloc[train_idx], X_train.iloc[val_idx]
    yt, yv = y_train.iloc[train_idx], y_train.iloc[val_idx]
    fold_pipe = clone(xgb_best_pipeline)
    fold_pipe.fit(Xt, yt)
    oof_probas[val_idx] = fold_pipe.predict_proba(Xv)[:, 1]
    oof_preds_05[val_idx] = fold_pipe.predict(Xv)

# Precision-Recall curve
precisions, recalls, thresholds = precision_recall_curve(y_train, oof_probas)

# Sweep thresholds and compute F1 at each
f1_scores_sweep = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

# Find the threshold that maximises F1
best_f1_idx = np.argmax(f1_scores_sweep)
best_threshold_f1 = thresholds[best_f1_idx]
print(f'Best threshold (max F1): {best_threshold_f1:.4f}')
print(f'  F1={f1_scores_sweep[best_f1_idx]:.3f}, precision={precisions[best_f1_idx]:.3f}, recall={recalls[best_f1_idx]:.3f}')

# For a safety system, also find the threshold that gets recall >= 0.80
# (catching at least 80% of real stops)
safety_mask = recalls[:-1] >= 0.80
if safety_mask.any():
    # Among thresholds with recall >= 0.80, pick the one with highest precision
    safety_idx = np.where(safety_mask)[0][np.argmax(precisions[:-1][safety_mask])]
    safety_threshold = thresholds[safety_idx]
    print(f'\nSafety threshold (recall >= 0.80): {safety_threshold:.4f}')
    print(f'  F1={f1_scores_sweep[safety_idx]:.3f}, precision={precisions[safety_idx]:.3f}, recall={recalls[safety_idx]:.3f}')
else:
    safety_threshold = best_threshold_f1
    print('\nNo threshold achieves recall >= 0.80')

# Plot precision-recall tradeoff
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PR curve
axes[0].plot(recalls, precisions, 'b-', linewidth=2)
axes[0].axvline(recalls[best_f1_idx], color='red', linestyle='--', alpha=0.7, label=f'Best F1 threshold ({best_threshold_f1:.3f})')
if safety_mask.any():
    axes[0].axvline(recalls[safety_idx], color='green', linestyle='--', alpha=0.7, label=f'Safety threshold ({safety_threshold:.3f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve (XGBoost tuned, OOF)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: F1 vs threshold
axes[1].plot(thresholds, f1_scores_sweep, 'g-', linewidth=2)
axes[1].axvline(best_threshold_f1, color='red', linestyle='--', alpha=0.7, label=f'Best F1={f1_scores_sweep[best_f1_idx]:.3f}')
axes[1].axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 vs Decision Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/fig_threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table: default (0.5) vs best-F1 vs safety threshold
print('\n=== Threshold comparison (XGBoost tuned, out-of-fold) ===')
for label, thr in [('Default 0.5', 0.5), ('Best F1', best_threshold_f1), ('Safety', safety_threshold)]:
    preds = (oof_probas >= thr).astype(int)
    r = recall_score(y_train, preds)
    p = precision_score(y_train, preds, zero_division=0)
    f = f1_score(y_train, preds, zero_division=0)
    ba = balanced_accuracy_score(y_train, preds)
    print(f'  {label:12s} (t={thr:.3f}): recall={r:.3f}, precision={p:.3f}, F1={f:.3f}, bal_acc={ba:.3f}')


## Baseline vs tuned comparison

TODO: side-by-side table, one row pair (baseline, tuned) per model.

In [ ]:
results.sort_values("pr_auc_mean", ascending=False)

## Decision log

### Decision (Meegasthanna): Logistic Regression Hyperparameter Tuning
- **What we tuned:** Regularization strength `C` (log-uniform distribution across $10^{-4}$ to $10^{2}$), regularization penalty (`l1` Lasso vs `l2` Ridge), and `class_weight` options.
- **Evidence:** `RandomizedSearchCV` with 35 iterations using 5-fold `StratifiedGroupKFold` CV (grouped by cycle). Tuned PR-AUC increased from **0.111** (baseline) to **0.165** with optimal L2 penalty and tuned `C`.
- **Alternative considered:** ElasticNet with `saga` solver.
- **Why rejected:** `liblinear` with L1/L2 provided fast, stable convergence without convergence warnings and achieved equivalent ranking performance.
- **Key Insight for Evaluation 2 Viva:** While hyperparameter tuning boosted PR-AUC by ~48% relative to baseline Logistic Regression, the linear model's PR-AUC (0.165) remains substantially below tree-based ensembles (XGBoost 0.418, Random Forest 0.428). This empirically proves that protective-stop fault signatures involve non-linear, multi-axis joint interactions that linear decision boundaries cannot capture.

### Decision (Bandara): XGBoost tuning — PR-AUC 0.418→0.473, recall 0.307→0.689
- **Evidence:** `RandomizedSearchCV` (40 iterations, `scoring="average_precision"`, same `StratifiedGroupKFold` folds as the baseline) found `scale_pos_weight≈28.2` as the dominant lever — notably close to the true 25.5:1 majority-to-minority ratio confirmed in notebook 01 (7,077/278), not a value picked at random. This drove recall from 0.307 to 0.689, at the cost of precision (0.552→0.441).
- **Alternative considered:** Leaving `scale_pos_weight` at the notebook-05 baseline default (1) and tuning only `n_estimators`/`max_depth`/`learning_rate`.
- **Why rejected:** Recall is the metric that matters most for a safety system — missing a real protective stop is far costlier than an extra false alarm — so a tuning search that let `scale_pos_weight` move toward the data's actual imbalance ratio was the right choice, even though it costs some precision.

### Decision (Bandara): Random Forest tuning — PR-AUC 0.428→0.439, recall 0.161→0.499
- **Evidence:** Same search setup; found `class_weight="balanced_subsample"` (not plain `"balanced"`), `max_depth=15`, `min_samples_leaf=7`. Recall roughly tripled (0.161→0.499) versus the baseline, though it still trails XGBoost's tuned recall by a wide margin (0.499 vs 0.689).
- **Alternative considered:** Restricting the search to `class_weight="balanced"` only (matching the baseline setting) rather than including `"balanced_subsample"` in the search space.
- **Why rejected:** `balanced_subsample` recomputes class weights per bootstrap sample rather than once globally, which the search preferred — worth including both options rather than assuming the baseline's default was already best.

### Decision (Bandara): XGBoost over Random Forest and Logistic Regression, after tuning
- **Evidence:** Tuned XGBoost leads on both PR-AUC (0.473 vs 0.439 RF vs 0.165 LogReg) and recall (0.689 vs 0.499 RF vs LogReg's own high-recall/low-precision profile) — not a close call, and consistent with Meegasthanna's finding that a linear decision boundary can't capture the multi-axis joint interactions this problem needs.
- **Alternative considered:** Waiting for SVM's tuned results (still pending) before drawing any conclusion.
- **Why noted now, not rejected:** This isn't the final model decision — that's notebook 07, after every model is tuned and the whole team compares at the 29 Sep results meeting. But it's worth flagging early: XGBoost is the front-runner so far by a clear margin, not a marginal one.

### Decision (Umer): SVM (RBF) tuning -- C, gamma, class_weight search
- **What we tuned:** `C` (loguniform 0.01 to 100), `gamma` (loguniform 1e-4 to 1), and `class_weight` (balanced vs None). 40 iterations of `RandomizedSearchCV`, same `StratifiedGroupKFold` folds as all other models.
- **Evidence:** The search explored whether SVM benefits more from wider margin (low C) or tighter fit (high C), and whether the RBF kernel width (gamma) needs to be narrower or wider than sklearn's default `1/n_features`.
- **Alternative considered:** Adding polynomial kernel to the search space (`kernel` as a hyperparameter).
- **Why rejected:** The RBF kernel is a universal approximator for SVMs on moderate-dimensional data (16 features). Adding the polynomial kernel would double the search space without strong prior evidence it would outperform RBF, and the 40-iteration budget is already tight given SVM's training cost with `probability=True`.

### Decision (Umer): SMOTE vs class weights -- class weights win for this dataset
- **Evidence:** Compared three configurations on the leading model (XGBoost): (1) tuned `scale_pos_weight` (from the tuning search), (2) SMOTE with default XGBoost, (3) SMOTE with tuned XGBoost hyperparams but without `scale_pos_weight`. SMOTE resampling was done inside each CV fold (using `imblearn.pipeline.Pipeline`) to avoid leaking synthetic data into validation folds.
- **Alternative considered:** SMOTE with `sampling_strategy` < 1.0 (partial oversampling) or SMOTE + Tomek links for hybrid resampling.
- **Why rejected:** The `scale_pos_weight` approach already achieved strong PR-AUC and recall without the computational overhead and risk of overfitting that SMOTE introduces by creating synthetic minority samples. For this small dataset (~170 positive examples in training), generating synthetic data from limited positive examples can produce unrealistic feature combinations that hurt generalization.

### Decision (Umer): Decision-threshold tuning -- operating point for a safety system
- **Evidence:** Swept thresholds on out-of-fold predictions from the best model (tuned XGBoost) across the same CV folds. The default 0.5 threshold is not optimal for this 3.1%-positive dataset -- the model's predicted probabilities cluster well below 0.5 for most positive examples.
- **Key finding:** A lower threshold recovers substantially more real stops (higher recall) at the cost of more false alarms. For a safety-critical system, missing a real protective stop is far costlier than a false alarm, so the operating point should prioritize recall.
- **Alternative considered:** Using the threshold that maximizes F1 (balanced precision-recall trade-off).
- **Why both are reported:** The best-F1 threshold is the statistically optimal balanced point, but for deployment in a safety system, the team may choose the safety threshold (recall >= 0.80) even at lower precision. This is a business decision, not a purely statistical one -- flagged for the 29 Sep results meeting.